# 07A – SHAP Global Explainability (Enterprise)

## Business Objective
Understand **which financial features most influence bankruptcy predictions** at a global level using SHAP (SHapley Additive exPlanations). This notebook helps validate the model, improve transparency, and provide business stakeholders with interpretable insights.


In [ ]:
import pandas as pd
import numpy as np
import shap
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# Load trained production model
model = joblib.load("production_bankruptcy_model.joblib")

# Load data
df = pd.read_csv("american_bankruptcy.csv")
df["target"] = df["status_label"].map({"alive":0,"failed":1})

drop_cols=["status_label","target"]
if "company_name" in df.columns:
    drop_cols.append("company_name")

X=df.drop(columns=drop_cols)
y=df["target"]

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

display(X_test.head())
print("Test shape:", X_test.shape)


## Create SHAP Explainer

In [ ]:
# Sample to reduce computation time
X_sample = X_test.sample(min(200, len(X_test)), random_state=42)

explainer = shap.Explainer(model.predict, X_sample)
shap_values = explainer(X_sample)

print("SHAP values computed successfully.")


## SHAP Summary (Beeswarm)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20)
plt.show()


## SHAP Bar Plot

In [ ]:
shap.plots.bar(shap_values, max_display=20)
plt.show()


## Global Feature Importance Table

In [ ]:
importance = pd.DataFrame({
    "Feature": X_sample.columns,
    "MeanAbsSHAP": np.abs(shap_values.values).mean(axis=0)
}).sort_values("MeanAbsSHAP", ascending=False)

display(importance)
importance.to_csv("shap_global_feature_importance.csv", index=False)


## Top 10 Bankruptcy Drivers

In [ ]:
display(importance.head(10))

plt.figure(figsize=(8,5))
plt.barh(
    importance.head(10)["Feature"][::-1],
    importance.head(10)["MeanAbsSHAP"][::-1]
)
plt.title("Top 10 Global Bankruptcy Drivers")
plt.tight_layout()
plt.show()


## Business Interpretation

In [ ]:
top = importance.head(5)

print("Top risk-driving features:")
for _, row in top.iterrows():
    print(f"- {row['Feature']} (Mean |SHAP| = {row['MeanAbsSHAP']:.4f})")

print("\nInterpretation:")
print("- Higher SHAP magnitude indicates greater influence on bankruptcy prediction.")
print("- Review these variables with finance stakeholders for actionable risk monitoring.")


## Executive Summary

In [ ]:
print("✔ Global explainability completed.")
print("✔ SHAP feature rankings exported to 'shap_global_feature_importance.csv'.")
print("✔ Proceed to Notebook 07B for local explanations of individual companies.")
